In [1]:
# 设置环境变量
%env HF_ENDPOINT=https://hf-mirror.com
%env HF_HOME=/root/autodl-tmp/hf

env: HF_ENDPOINT=https://hf-mirror.com
env: HF_HOME=/root/autodl-tmp/hf


In [3]:
# 安装依赖库
%pip install trl
%pip install peft
%pip install bitsandbytes
%pip install unsloth

Looking in indexes: http://mirrors.aliyun.com/pypi/simple
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: http://mirrors.aliyun.com/pypi/simple
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: http://mirrors.aliyun.com/pypi/simple
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: http://mirrors.aliyun.com/pypi/simple
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# 加载model和tokenizer

from unsloth import FastLanguageModel


model_name = "Qwen/Qwen3-8B"
max_length = 2048  # 支持自动 RoPE Scaling，因此可以自行设置长度

# 加载模型
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_length,
    dtype=None,  # 自动检测。Tesla T4/V100 通常用 Float16，Ampere+ 通常用 Bfloat16
    load_in_4bit=True,  # 使用 4bit 量化降低显存占用，也可以设为 False
)



# 对模型打补丁并添加高速 LoRA 权重
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,  # 当前对 dropout=0 做了优化
    bias="none",  # 当前对 bias="none" 做了优化
    use_gradient_checkpointing=True,
    random_state=3407,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.357 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
# 数据集
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset

dataset_dict = load_dataset('json',data_files={"train":"datas/keywords_data_train.jsonl","test":"datas/keywords_data_test.jsonl"})

# 将数据转为标准对话格式（OpenAI）
def map_func(example):
    conversation = example['conversation']
    messages = []
    for item in conversation:
        messages.append({'role':'user','content':item['human']})
        messages.append({'role':'assistant','content':item['assistant']})
    return {'messages':messages}

dataset_dict = dataset_dict.map(map_func,batched=False,remove_columns=['dataset','conversation','category','conversation_id'])


# 将对话格式的数据转为字符串（Chat Templete）
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen3", # change this to the right chat_template name
)
def formatting_prompts_func(examples):
   convos = examples["messages"]
   texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
   return {'text':texts}

dataset_dict = dataset_dict.map(formatting_prompts_func,batched=True,remove_columns=['messages'])

In [ ]:
from trl import SFTConfig,SFTTrainer

# Configure trainer
training_args = SFTConfig(
    output_dir="/root/autodl-tmp/sft/Qwen3-8B/sft-unsloth",
    max_steps=1000,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    eval_strategy="steps",
    eval_steps=100,
    load_best_model_at_end=True,
    bf16=True,
    warmup_steps=50,
    logging_dir='/root/tf-logs',
    report_to=["tensorboard"],
)

# Initialize trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset_dict["train"],
    eval_dataset=dataset_dict["test"],
    processing_class=tokenizer
)

In [ ]:
# 察看数据集处理结果
dataloader = trainer.get_train_dataloader()
batch = next(iter(dataloader))
print(tokenizer.decode(batch['input_ids'][0]))

In [ ]:
# 开始训练
trainer.train()

In [ ]:
# 保存模型
trainer.save_model('/root/autodl-tmp/sft/Qwen3-8B/sft-unsloth/best')

In [ ]:
# 保存完整模型
model.save_pretrained_merged("/root/autodl-tmp/sft/Qwen3-8B/sft-unsloth/merged", tokenizer, save_method = "merged_16bit")